In [94]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import ta
from torch.utils.data import Dataset, DataLoader

In [117]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [136]:
parent_dir = os.path.dirname(os.getcwd())
df = pd.read_csv(parent_dir + '/data/sp500_stocks.csv')

In [138]:
def compute_technical_indicators(df, ticker_list=None):
    """
    Compute required technical indicators:
    - RSI(14)
    - MACD(12,26,9)
    - Bollinger Band width(20)
    - ATR(14)
    - OBV
    - 5-day and 20-day moving average crossover signal
    """
    if ticker_list is None:
        ticker_list = df['Ticker'].unique()
    
    df = df.copy()
    df = df.sort_values(['Ticker', 'Date'])
    
    # Define indicator columns
    indicator_cols = [
        'RSI_14', 'MACD_line', 'MACD_signal', 'MACD_histogram',
        'BB_width', 'ATR_14', 'OBV', 'MA_Crossover_5_20'
    ]
    
    for col in indicator_cols:
        df[col] = np.nan
    
    print(f"Processing {len(ticker_list)} tickers...")
    
    for idx, ticker in enumerate(ticker_list):
        if idx % 10 == 0:
            print(f"  Processing ticker {idx+1}/{len(ticker_list)}: {ticker}")
        
        mask = df['Ticker'] == ticker
        ticker_indices = df[mask].index
        
        close = df.loc[mask, 'Close'].values
        high = df.loc[mask, 'High'].values
        low = df.loc[mask, 'Low'].values
        volume = df.loc[mask, 'Volume'].values
        
        close_series = pd.Series(close, index=ticker_indices)
        high_series = pd.Series(high, index=ticker_indices)
        low_series = pd.Series(low, index=ticker_indices)
        volume_series = pd.Series(volume, index=ticker_indices)
        
        # 1. RSI (14-day)
        rsi = ta.momentum.RSIIndicator(close_series, window=14).rsi()
        df.loc[mask, 'RSI_14'] = rsi.values
        
        # 2. MACD (12, 26, 9)
        macd = ta.trend.MACD(close_series)
        df.loc[mask, 'MACD_line'] = macd.macd().values
        df.loc[mask, 'MACD_signal'] = macd.macd_signal().values
        df.loc[mask, 'MACD_histogram'] = macd.macd_diff().values
        
        # 3. Bollinger Band Width (20-day, 2 std dev)
        bb = ta.volatility.BollingerBands(close_series, window=20, window_dev=2)
        bb_upper = bb.bollinger_hband().values
        bb_lower = bb.bollinger_lband().values
        bb_middle = bb.bollinger_mavg().values
        df.loc[mask, 'BB_width'] = (bb_upper - bb_lower) / bb_middle
        
        # 4. ATR (14-day)
        atr = ta.volatility.AverageTrueRange(high_series, low_series, close_series, window=14)
        df.loc[mask, 'ATR_14'] = atr.average_true_range().values
        
        # 5. OBV (On-Balance Volume)
        obv = ta.volume.OnBalanceVolumeIndicator(close_series, volume_series)
        df.loc[mask, 'OBV'] = obv.on_balance_volume().values
        
        # 6. MA Crossover (5-day vs 20-day)
        sma_5 = close_series.rolling(window=5).mean().values
        sma_20 = close_series.rolling(window=20).mean().values
        df.loc[mask, 'MA_Crossover_5_20'] = sma_5 - sma_20
    
    feature_cols = [
        'RSI_14', 'MACD_line', 'MACD_signal', 'MACD_histogram',
        'BB_width', 'ATR_14', 'OBV', 'MA_Crossover_5_20'
    ]
    
    return df, feature_cols

# Assuming 'df' is your raw data with columns: Date, Open, High, Low, Close, Volume, Ticker
ticker_list = df['Ticker'].unique().tolist()
print(f"Found {len(ticker_list)} tickers")

df_with_features, feature_cols = compute_technical_indicators(df, ticker_list)

print(f"\n✅ Added {len(feature_cols)} technical indicator features")
print(f"Features: {feature_cols}")
print(f"Data shape: {df_with_features.shape}")

Found 50 tickers
Processing 50 tickers...
  Processing ticker 1/50: AAPL
  Processing ticker 11/50: GS
  Processing ticker 21/50: MRK
  Processing ticker 31/50: CVX
  Processing ticker 41/50: CMCSA

✅ Added 8 technical indicator features
Features: ['RSI_14', 'MACD_line', 'MACD_signal', 'MACD_histogram', 'BB_width', 'ATR_14', 'OBV', 'MA_Crossover_5_20']
Data shape: (259073, 15)


In [139]:
# ============================================
# 2. CREATE TARGET VARIABLES
# ============================================

print("\n" + "="*60)
print("2. CREATING TARGET VARIABLES")
print("="*60)

def create_targets(df):
    """
    Create target variables:
    - Regression: next-day log return
    - Classification: 5-day return direction (1=up, 0=down)
    """
    df = df.copy()
    df = df.sort_values(['Ticker', 'Date'])
    
    for ticker in df['Ticker'].unique():
        mask = df['Ticker'] == ticker
        close = df.loc[mask, 'Close']
        
        # Regression: next-day log return
        df.loc[mask, 'Target_Regression'] = np.log(close.shift(-1) / close)
        
        # Classification: 5-day direction
        forward_5_return = close.shift(-5) / close - 1
        df.loc[mask, 'Target_Classification'] = (forward_5_return > 0).astype(int)
    
    return df

df_with_features = create_targets(df_with_features)

print(f"✅ Created target variables")
print(f"  Regression: next-day log return")
print(f"  Classification: 5-day direction")

# Check target statistics
regression_target = df_with_features['Target_Regression'].dropna()
print(f"\nRegression Target Statistics:")
print(f"  Mean: {regression_target.mean():.6f}")
print(f"  Std: {regression_target.std():.6f}")

classification_target = df_with_features['Target_Classification'].dropna()
print(f"\nClassification Target Statistics:")
print(f"  Up days: {classification_target.sum():.0f} ({classification_target.mean():.1%})")
print(f"  Down days: {(1 - classification_target).sum():.0f} ({1 - classification_target.mean():.1%})")


2. CREATING TARGET VARIABLES
✅ Created target variables
  Regression: next-day log return
  Classification: 5-day direction

Regression Target Statistics:
  Mean: 0.000478
  Std: 0.019805

Classification Target Statistics:
  Up days: 142543 (55.0%)
  Down days: 116530 (45.0%)


In [142]:
print("\n" + "="*60)
print("3. HANDLING NaN VALUES (Warm-up Period)")
print("="*60)

# Check missing values before handling
missing_before = df_with_features[feature_cols + ['Target_Regression']].isna().sum()
print(f"Missing values before handling:")
print(f"  Total: {missing_before.sum()}")
print(f"  Columns with NaN: {missing_before[missing_before > 0].index.tolist()}")

# Drop rows with NaN in features or target
df_final = df_with_features.dropna(subset=feature_cols + ['Target_Regression'])

print(f"\n✅ After dropping NaN:")
print(f"  Original shape: {df_with_features.shape}")
print(f"  Final shape: {df_final.shape}")
print(f"  Rows dropped: {len(df_with_features) - len(df_final)}")
print(f"  Remaining missing: {df_final[feature_cols + ['Target_Regression']].isna().sum().sum()}")


3. HANDLING NaN VALUES (Warm-up Period)
Missing values before handling:
  Total: 7150
  Columns with NaN: ['RSI_14', 'MACD_line', 'MACD_signal', 'MACD_histogram', 'BB_width', 'MA_Crossover_5_20', 'Target_Regression']

✅ After dropping NaN:
  Original shape: (259073, 17)
  Final shape: (257373, 17)
  Rows dropped: 1700
  Remaining missing: 0


In [152]:
def prepare_data_final(df, ticker_list, feature_cols, train_ratio=0.7, val_ratio=0.15):
    """
    FINAL APPROACH:
    1. Split temporally (NO SHUFFLING)
    2. Log transform OBV
    3. StandardScaler fit on TRAINING only
    """
    
    print(f"\nProcessing {len(ticker_list)} tickers...")
    
    # Initialize lists to store data from all tickers
    all_X_train = []
    all_X_val = []
    all_X_test = []
    all_y_train_reg = []
    all_y_val_reg = []
    all_y_test_reg = []
    all_y_train_cls = []
    all_y_val_cls = []
    all_y_test_cls = []
    all_train_tickers = []
    all_val_tickers = []
    all_test_tickers = []
    all_train_dates = []
    all_val_dates = []
    all_test_dates = []
    
    # Step 1: Split data temporally (NO SHUFFLING!)
    for ticker in ticker_list:
        ticker_df = df[df['Ticker'] == ticker].sort_values('Date')
        
        # Get features and targets
        X = ticker_df[feature_cols].values
        y_reg = ticker_df['Target_Regression'].values
        y_cls = ticker_df['Target_Classification'].values
        dates = ticker_df['Date'].values
        
        n = len(X)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        
        # Split WITHOUT shuffling (maintain temporal order)
        X_train_raw = X[:train_end]
        X_val_raw = X[train_end:val_end]
        X_test_raw = X[val_end:]
        y_train_reg = y_reg[:train_end]
        y_val_reg = y_reg[train_end:val_end]
        y_test_reg = y_reg[val_end:]
        y_train_cls = y_cls[:train_end]
        y_val_cls = y_cls[train_end:val_end]
        y_test_cls = y_cls[val_end:]
        dates_train = dates[:train_end]
        dates_val = dates[train_end:val_end]
        dates_test = dates[val_end:]
        
        # Append to lists
        all_X_train.append(X_train_raw)
        all_X_val.append(X_val_raw)
        all_X_test.append(X_test_raw)
        all_y_train_reg.append(y_train_reg)
        all_y_val_reg.append(y_val_reg)
        all_y_test_reg.append(y_test_reg)
        all_y_train_cls.append(y_train_cls)
        all_y_val_cls.append(y_val_cls)
        all_y_test_cls.append(y_test_cls)
        all_train_tickers.extend([ticker] * len(y_train_reg))
        all_val_tickers.extend([ticker] * len(y_val_reg))
        all_test_tickers.extend([ticker] * len(y_test_reg))
        all_train_dates.extend(dates_train)
        all_val_dates.extend(dates_val)
        all_test_dates.extend(dates_test)
    
    # Step 2: Concatenate all data
    X_train_raw = np.vstack(all_X_train)
    X_val_raw = np.vstack(all_X_val)
    X_test_raw = np.vstack(all_X_test)
    y_train_reg = np.concatenate(all_y_train_reg)
    y_val_reg = np.concatenate(all_y_val_reg)
    y_test_reg = np.concatenate(all_y_test_reg)
    y_train_cls = np.concatenate(all_y_train_cls)
    y_val_cls = np.concatenate(all_y_val_cls)
    y_test_cls = np.concatenate(all_y_test_cls)
    
    print(f"\nRaw data shapes (BEFORE normalization):")
    print(f"  X_train_raw: {X_train_raw.shape}")
    print(f"  X_val_raw: {X_val_raw.shape}")
    print(f"  X_test_raw: {X_test_raw.shape}")
    
    # Step 3: Log transform OBV (handle extreme values)
    obv_idx = feature_cols.index('OBV')
    print(f"\nOBV index: {obv_idx} (feature: OBV)")
    
    # Calculate shift from training data only (NO LOOK-AHEAD!)
    obv_train = X_train_raw[:, obv_idx]
    obv_shift = obv_train.min() - 1 if obv_train.min() < 0 else 0
    
    print(f"OBV shift value: {obv_shift:.2f}")
    
    # Apply log transform to ALL splits using training shift
    X_train_log = X_train_raw.copy()
    X_val_log = X_val_raw.copy()
    X_test_log = X_test_raw.copy()
    
    # Log transform OBV: log(x - shift + 1)
    X_train_log[:, obv_idx] = np.log(obv_train - obv_shift + 1)
    X_val_log[:, obv_idx] = np.log(X_val_raw[:, obv_idx] - obv_shift + 1)
    X_test_log[:, obv_idx] = np.log(X_test_raw[:, obv_idx] - obv_shift + 1)
    
    print(f"\nOBV after log transform:")
    print(f"  Train - Mean: {X_train_log[:, obv_idx].mean():.4f}, Std: {X_train_log[:, obv_idx].std():.4f}")
    print(f"  Val   - Mean: {X_val_log[:, obv_idx].mean():.4f}, Std: {X_val_log[:, obv_idx].std():.4f}")
    print(f"  Test  - Mean: {X_test_log[:, obv_idx].mean():.4f}, Std: {X_test_log[:, obv_idx].std():.4f}")
    
    # Step 4: Fit StandardScaler on TRAINING data ONLY
    print(f"\nFitting StandardScaler on TRAINING data only...")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_log)
    X_val_scaled = scaler.transform(X_val_log)
    X_test_scaled = scaler.transform(X_test_log)
    
    print(f"\nAfter StandardScaler:")
    print(f"  X_train - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
    print(f"  X_val   - Mean: {X_val_scaled.mean():.6f}, Std: {X_val_scaled.std():.6f}")
    print(f"  X_test  - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")
    
    # Return all data
    return {
        'X_train': X_train_scaled,
        'X_val': X_val_scaled,
        'X_test': X_test_scaled,
        'y_train_reg': y_train_reg,
        'y_val_reg': y_val_reg,
        'y_test_reg': y_test_reg,
        'y_train_cls': y_train_cls,
        'y_val_cls': y_val_cls,
        'y_test_cls': y_test_cls,
        'train_tickers': np.array(all_train_tickers),
        'val_tickers': np.array(all_val_tickers),
        'test_tickers': np.array(all_test_tickers),
        'train_dates': np.array(all_train_dates, dtype='datetime64'),
        'val_dates': np.array(all_val_dates, dtype='datetime64'),
        'test_dates': np.array(all_test_dates, dtype='datetime64'),
        'scaler': scaler,
        'obv_shift': obv_shift,
        'feature_cols': feature_cols
    }

# ============================================
# RUN THE FINAL APPROACH
# ============================================

# Get ticker list
ticker_list = df_final['Ticker'].unique().tolist()

# Prepare data
ticker_data_final = prepare_data_final(df_final, ticker_list, feature_cols)

# ============================================
# VERIFY DATA
# ============================================

print("\n" + "="*60)
print("VERIFYING DATA")
print("="*60)

print(f"\nFinal data shapes:")
print(f"  X_train: {ticker_data_final['X_train'].shape}")
print(f"  X_val: {ticker_data_final['X_val'].shape}")
print(f"  X_test: {ticker_data_final['X_test'].shape}")

print(f"\nFinal data statistics:")
print(f"  X_train - Mean: {ticker_data_final['X_train'].mean():.6f}, Std: {ticker_data_final['X_train'].std():.6f}")
print(f"  X_val   - Mean: {ticker_data_final['X_val'].mean():.6f}, Std: {ticker_data_final['X_val'].std():.6f}")
print(f"  X_test  - Mean: {ticker_data_final['X_test'].mean():.6f}, Std: {ticker_data_final['X_test'].std():.6f}")

# Check for NaN
print(f"\nChecking for NaN:")
print(f"  X_train - NaN: {np.isnan(ticker_data_final['X_train']).any()}")
print(f"  X_val   - NaN: {np.isnan(ticker_data_final['X_val']).any()}")
print(f"  X_test  - NaN: {np.isnan(ticker_data_final['X_test']).any()}")

# ============================================
# SAVE DATA
# ============================================

print("\n" + "="*60)
print("SAVING DATA")
print("="*60)

os.makedirs('data/processed', exist_ok=True)

np.savez_compressed(
    'data/processed/all_tickers_preprocessed_final.npz',
    X_train=ticker_data_final['X_train'],
    X_val=ticker_data_final['X_val'],
    X_test=ticker_data_final['X_test'],
    y_train_reg=ticker_data_final['y_train_reg'],
    y_val_reg=ticker_data_final['y_val_reg'],
    y_test_reg=ticker_data_final['y_test_reg'],
    y_train_cls=ticker_data_final['y_train_cls'],
    y_val_cls=ticker_data_final['y_val_cls'],
    y_test_cls=ticker_data_final['y_test_cls'],
    train_tickers=ticker_data_final['train_tickers'],
    val_tickers=ticker_data_final['val_tickers'],
    test_tickers=ticker_data_final['test_tickers'],
    train_dates=ticker_data_final['train_dates'],
    val_dates=ticker_data_final['val_dates'],
    test_dates=ticker_data_final['test_dates']
)

joblib.dump(ticker_data_final['scaler'], 'data/processed/all_tickers_scaler_final.pkl')

metadata = {
    'tickers': ticker_list,
    'num_tickers': len(ticker_list),
    'feature_cols': feature_cols,
    'num_features': len(feature_cols),
    'train_shape': ticker_data_final['X_train'].shape,
    'val_shape': ticker_data_final['X_val'].shape,
    'test_shape': ticker_data_final['X_test'].shape,
    'num_train': len(ticker_data_final['y_train_reg']),
    'num_val': len(ticker_data_final['y_val_reg']),
    'num_test': len(ticker_data_final['y_test_reg']),
    'train_mean': ticker_data_final['X_train'].mean(),
    'train_std': ticker_data_final['X_train'].std(),
    'val_mean': ticker_data_final['X_val'].mean(),
    'val_std': ticker_data_final['X_val'].std(),
    'test_mean': ticker_data_final['X_test'].mean(),
    'test_std': ticker_data_final['X_test'].std(),
    'obv_shift': ticker_data_final['obv_shift'],
    'scaler_type': 'StandardScaler (fit on training only)',
    'log_transform_applied': 'OBV',
    'split_ratio': '70/15/15',
    'temporal_split': True,
    'no_shuffle': True,
    'date_saved': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}
joblib.dump(metadata, 'data/processed/all_tickers_metadata_final.pkl')

print(f"\n✅ DATA SAVED SUCCESSFULLY!")
print(f"  File: data/processed/all_tickers_preprocessed_final.npz")
print(f"  Scaler: data/processed/all_tickers_scaler_final.pkl")
print(f"  Metadata: data/processed/all_tickers_metadata_final.pkl")

print("\n" + "-"*60)
print("CURRICULUM REQUIREMENTS CHECK:")
print("-"*60)
print("✅ 1. Technical indicators computed")
print("✅ 2. Target variables created")
print("✅ 3. Temporal split (NO SHUFFLING) - 70/15/15")
print("✅ 4. Log transform applied to OBV")
print("✅ 5. StandardScaler fit on TRAINING data only")
print("✅ 6. Same scaler applied to validation and test")
print("✅ 7. NaN values handled")
print("✅ 8. Data saved")

print("\n" + "="*60)
print("✅ PREPROCESSING COMPLETE!")
print("="*60)


Processing 50 tickers...

Raw data shapes (BEFORE normalization):
  X_train_raw: (180132, 8)
  X_val_raw: (38596, 8)
  X_test_raw: (38645, 8)

OBV index: 6 (feature: OBV)
OBV shift value: -4540063001.00

OBV after log transform:
  Train - Mean: 22.5286, Std: 0.5996
  Val   - Mean: 22.7147, Std: 0.7249
  Test  - Mean: 22.7426, Std: 0.7433

Fitting StandardScaler on TRAINING data only...

After StandardScaler:
  X_train - Mean: -0.000000, Std: 1.000000
  X_val   - Mean: 0.376663, Std: 2.610474
  X_test  - Mean: 0.679225, Std: 3.596564

VERIFYING DATA

Final data shapes:
  X_train: (180132, 8)
  X_val: (38596, 8)
  X_test: (38645, 8)

Final data statistics:
  X_train - Mean: -0.000000, Std: 1.000000
  X_val   - Mean: 0.376663, Std: 2.610474
  X_test  - Mean: 0.679225, Std: 3.596564

Checking for NaN:
  X_train - NaN: False
  X_val   - NaN: False
  X_test  - NaN: False

SAVING DATA

✅ DATA SAVED SUCCESSFULLY!
  File: data/processed/all_tickers_preprocessed_final.npz
  Scaler: data/process

## Why no random splits for financial data:
- Financial data is time-series, which means that each observation depends on previous ones
- Shuffling introduces look-ahead bias, causing future data leaks into training
- Random splits give unrealistically optimistic results


In [ ]:
# ============================================
# 5. CREATE PYTORCH DATASET AND DATALOADER
# ============================================

print("\n" + "="*60)
print("5. CREATING PYTORCH DATASET & DATALOADER")
print("="*60)

class StockDataset(Dataset):
    """PyTorch Dataset for stock data"""
    def __init__(self, features, targets):
        self.X = torch.FloatTensor(features)
        self.y = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = StockDataset(ticker_data['X_train'], ticker_data['y_train_reg'])
val_dataset = StockDataset(ticker_data['X_val'], ticker_data['y_val_reg'])
test_dataset = StockDataset(ticker_data['X_test'], ticker_data['y_test_reg'])

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✅ DataLoaders created with batch_size={BATCH_SIZE}")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")
print(f"\n  Train shuffle: True (for training)")
print(f"  Val shuffle: False (maintains temporal order)")
print(f"  Test shuffle: False (maintains temporal order)")

In [126]:
os.makedirs('data/processed', exist_ok=True)

np.savez_compressed(
    'data/processed/all_tickers_preprocessed_fixed.npz',
    X_train=ticker_data['X_train'],
    X_val=ticker_data['X_val'],
    X_test=ticker_data['X_test'],
    y_train_reg=ticker_data['y_train_reg'],
    y_val_reg=ticker_data['y_val_reg'],
    y_test_reg=ticker_data['y_test_reg'],
    y_train_cls=ticker_data['y_train_cls'],
    y_val_cls=ticker_data['y_val_cls'],
    y_test_cls=ticker_data['y_test_cls'],
    train_tickers=ticker_data['train_tickers'],
    val_tickers=ticker_data['val_tickers'],
    test_tickers=ticker_data['test_tickers'],
    train_dates=ticker_data['train_dates'],
    val_dates=ticker_data['val_dates'],
    test_dates=ticker_data['test_dates']
)

joblib.dump(ticker_data['scaler'], 'data/processed/all_tickers_scaler_fixed.pkl')

metadata = {
    'tickers': ticker_list,
    'num_tickers': len(ticker_list),
    'feature_cols': ticker_data['feature_cols_original'],
    'num_features': len(ticker_data['feature_cols_original']),
    'train_shape': ticker_data['X_train'].shape,
    'val_shape': ticker_data['X_val'].shape,
    'test_shape': ticker_data['X_test'].shape,
    'num_train': len(ticker_data['y_train_reg']),
    'num_val': len(ticker_data['y_val_reg']),
    'num_test': len(ticker_data['y_test_reg']),
    'train_mean': ticker_data['X_train'].mean(),
    'train_std': ticker_data['X_train'].std(),
    'val_mean': ticker_data['X_val'].mean(),
    'val_std': ticker_data['X_val'].std(),
    'test_mean': ticker_data['X_test'].mean(),
    'test_std': ticker_data['X_test'].std(),
    'date_saved': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
}
joblib.dump(metadata, 'data/processed/all_tickers_metadata_fixed.pkl')

print(f"\n✅ DATA SAVED SUCCESSFULLY!")
print(f"  File: data/processed/all_tickers_preprocessed_fixed.npz")
print(f"  Scaler: data/processed/all_tickers_scaler_fixed.pkl")
print(f"  Metadata: data/processed/all_tickers_metadata_fixed.pkl")

print("\n" + "-"*60)
print("VERIFICATION:")
print("-"*60)
print(f"  Train - Mean: {ticker_data['X_train'].mean():.6f}, Std: {ticker_data['X_train'].std():.6f}")
print(f"  Val   - Mean: {ticker_data['X_val'].mean():.6f}, Std: {ticker_data['X_val'].std():.6f}")
print(f"  Test  - Mean: {ticker_data['X_test'].mean():.6f}, Std: {ticker_data['X_test'].std():.6f}")

if abs(ticker_data['X_val'].mean()) < 0.1 and abs(ticker_data['X_test'].mean()) < 0.1:
    print("\n🎉 SUCCESS! ALL data is properly normalized!")
else:
    print(f"\n⚠️ Still issues: Val mean={ticker_data['X_val'].mean():.6f}, Test mean={ticker_data['X_test'].mean():.6f}")

KeyError: 'feature_cols_original'